# Module 4: Human-in-the-Loop (HITL)

**Day 4 — LangGraph Agents, Memory, HITL & MCP**

## What you will learn
- **interrupt()**: pause execution at any node for human review
- **Command(resume=...)**: resume the graph after human approval
- **Approval workflow pattern**: submit -> review -> approve/reject
- **Breakpoints**: `interrupt_before` and `interrupt_after`

## Why HITL?
Fully autonomous agents make mistakes. In high-stakes domains, a human must review
before irreversible actions (send 5000 emails, approve Rs 50L loan, delete production data).

**Key insight**: `interrupt()` is not an error — it's a deliberate pause.


In [ ]:
import sys
sys.path.insert(0, '../src')
print('Path configured.')

## 1. The interrupt() Pattern

When the graph hits `interrupt()`, it pauses and stores its state in the checkpointer.
Your application code inspects the state, gets human input, then calls `graph.invoke(Command(resume=...))`.

In [ ]:
from day4.human_in_loop import build_hitl_graph, ApprovalWorkflow
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

# Mock agent: always drafts a high-stakes action
actions = [
    'Send promotional email to 50,000 Flipkart customers with 30% discount',
    'Approve home loan application #HL-2024-09150 for Rs 75,00,000',
    'Delete all transaction records older than 2 years from production DB',
]
call_count = [0]
def mock_agent(messages):
    call_count[0] += 1
    return actions[(call_count[0] - 1) % len(actions)]

checkpointer = MemorySaver()
graph = build_hitl_graph(mock_agent, checkpointer=checkpointer)
workflow = ApprovalWorkflow(graph)

print('HITL workflow built.')

## 2. Submit Request (graph pauses at interrupt)

In [ ]:
# Submit: graph runs until interrupt(), then pauses
draft = workflow.submit('Process bulk email campaign', 'session_001')

print('=' * 60)
print('DRAFT ACTION (awaiting human review):')
print('=' * 60)
print(f'{draft}')
print('=' * 60)
print()
print('Graph is now PAUSED. State is stored in checkpointer.')
print('Waiting for human decision...')

## 3. Human APPROVES

In [ ]:
# Human approves: graph resumes with Command(resume=True)
result = workflow.approve('session_001')

print('Human decision: APPROVED')
print(f'Result: {result}')

## 4. Human REJECTS

In [ ]:
# New session — this time human rejects
draft2 = workflow.submit('Delete production database backup', 'session_002')

print('DRAFT ACTION:')
print(f'  {draft2}')
print()

result2 = workflow.reject('session_002')
print('Human decision: REJECTED')
print(f'Result: {result2}')

## 5. Low-level interrupt() Pattern

Here's how the interrupt/resume pattern works under the hood:

In [ ]:
from langgraph.types import Command

# Manual interrupt/resume pattern:
config = {'configurable': {'thread_id': 'manual_demo'}}
call_count[0] = 0

# Step 1: Run until interrupt
graph.invoke(
    {
        'messages': [HumanMessage('Approve Rs 50L loan')],
        'draft_action': None,
        'approved': None,
        'final_result': None,
    },
    config=config
)

# Step 2: Inspect state
state = graph.get_state(config)
print('State after interrupt:')
print(f'  draft_action: {state.values.get("draft_action")}')
print(f'  approved:     {state.values.get("approved")}')
print(f'  Next node:    {state.next}')

# Step 3: Resume with human decision
result = graph.invoke(Command(resume=True), config=config)
print(f'\nAfter resume: {result.get("final_result")}')

## 6. HITL Use Cases in Production

In [ ]:
use_cases = [
    ('HDFC Bank',    'Loan officer reviews AI-recommended loan decision before approval'),
    ('Flipkart',     'Marketing manager approves AI-drafted bulk email before sending'),
    ('Infosys',      'Legal team reviews AI-drafted contracts before client submission'),
    ('Apollo',       'Doctor validates AI diagnosis before adding to patient record'),
    ('Swiggy',       'Support manager approves AI response before sending to customer'),
    ('Paytm',        'Compliance officer reviews large transaction alerts before blocking'),
]

print('HITL in Indian enterprise applications:')
print('=' * 60)
for company, use_case in use_cases:
    print(f'  [{company:12s}] {use_case}')

## Databricks Bridge

In [ ]:
# HITL + Databricks: store approval decisions in Delta Lake for audit
#
# def log_approval_decision(thread_id, action, approved, reviewer_id):
#     spark.createDataFrame([{
#         'thread_id':   thread_id,
#         'action':      action,
#         'approved':    approved,
#         'reviewer_id': reviewer_id,
#         'timestamp':   datetime.now(),
#     }]).write.format('delta').mode('append').save('/mnt/approval_audit/')
#
# # In your HITL workflow:
# draft = workflow.submit(request, thread_id)
# human_approved = get_human_input()  # from Slack, web app, etc.
# log_approval_decision(thread_id, draft, human_approved, reviewer_id)
# if human_approved:
#     result = workflow.approve(thread_id)
# else:
#     result = workflow.reject(thread_id)

print('Databricks: audit all HITL decisions in Delta Lake for compliance.')